# Chapter 4: Engineer the context layer

Companion to *Agent Harness Engineering*. Listings 4.1 to 4.4 live in `ahe/ch04.py`.

## Setup

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "ahe").exists():
    root = root.parent            # opened from inside chapter-0N/
sys.path.insert(0, str(root))
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
import os
LIVE = os.environ.get("AHE_LIVE") == "1"
print("Python", sys.version.split()[0], "| live calls:", "on" if LIVE else "off")

## 1. The exit test (Table 4.7)

In [ ]:
from ahe import ch03, ch04
from ahe.ch03 import request, run_task, Budget
run, mid = ch04.table_4_3()
print("naive:", ch04.size(ch04.naive_context(run)), "chars")
for row in ch04.exit_test(mid, ["n58", "n61"], (2400, 1200, 900, 500)):
    print(row)

## 2. What the model saw at one step

In [ ]:
from pprint import pprint
run = run_task(request("P042", 3), ch04.planner_v2, context=ch04.context_v2)
ctx = [e for e in run.events if e["kind"] == "context"][3]
print(ctx["sha"], ctx["chars"], "chars")
pprint(ctx["bundle"]["pending"])

## 3. Replay the three-context experiment (Table 4.4)

In [ ]:
from ahe import ch01, live
def replay(experiment, pid, context, n=5):
    replies = iter(r["reply"] for r in live.records(experiment))
    def planner(c):
        try:
            return ch01.parse_json(next(replies))
        except ValueError:
            return {"tool": None}
    return [run_task(request(pid, 3), planner, budget=Budget(steps=16, reads=6), context=context)
            for _ in range(n)]
for pid in ("P042", "P043"):
    for tag, context in (("naive", ch03.context_for), ("assembled", ch04.context_without_terms),
                         ("terms", ch04.context_v2)):
        runs = replay(f"ch04_{pid.lower()}_{tag}_gpt-6-luna", pid, context)
        done = sum(r.state == "completed" for r in runs)
        print(pid, f"{tag:10s} completed {done}/5, steps", [r.count("proposal") for r in runs])

### Live calls

Every live cell below does nothing unless `AHE_LIVE=1` is set before Jupyter starts. Settings come from the environment or from `~/.config/ahe/openrouter.env`:

```
AHE_BASE_URL=https://openrouter.ai/api/v1     # or https://<workspace>/serving-endpoints
AHE_API_KEY=...                                # never commit this file
AHE_MODEL=openai/gpt-6-luna
```

Any OpenAI-compatible endpoint works, including a Databricks workspace (base URL `https://<workspace-host>/serving-endpoints`, a personal access token as the key, and a served model name). `ahe.live.Recorder` replaces `chat` so that each call is appended to `runs/<experiment>.jsonl` with its prompt, reply, token counts, and cost. Use a new experiment name for your own runs so the book's records stay as they are.

In [ ]:
if LIVE:
    live.load_env()
    ch01.chat = live.Recorder("my_ch04_terms")
    r = run_task(request("P042", 3), ch03.live_planner, context=ch04.context_v2,
                 budget=Budget(steps=16, reads=6))
    print(r.state, r.reason, r.count("proposal"), "steps")
else:
    print("Set AHE_LIVE=1 to run the live planner on the assembled context.")

## Exercise solutions

### Exercise 4.1

`fake_planner` reads `context["observations"]`, which the assembled bundle does not have, so the run fails with a `KeyError`. It is a broken contract, not a bug on either side: the planner assumed the Chapter 3 schema. A planner is entitled to the fields a versioned context schema promises, and nothing else.

In [ ]:
r = run_task(request("P042", 3), ch03.fake_planner, context=ch04.context_v2)
print(r.state, r.reason)

### Exercise 4.2

The invariant: no sentence of any note appears in the bundle. Returning the quotation from `t_read` would put note text into observations and so into the recent window, which Section 2.5's contract keeps out: only grounded statements enter. A reviewer would see the same packet, but the log would show note text reaching the planner.

In [ ]:
import json
def no_note_text(run, ctx):
    text = json.dumps(ctx, default=str)
    return not any(n.text[:30] in text for n in ch04.NOTES[run.request["patient"]])
run = run_task(request("P042", 3), ch04.planner_v2, context=ch04.context_v2)
print(all(no_note_text(run, e["bundle"]) for e in run.events if e["kind"] == "context"))

### Exercise 4.3

Lexical search with the listed terms finds nothing, and the rule stays `unknown`. A fixture that justifies vector search would state that n70 is relevant, that a search for anticoagulants must return it, and that `ground` must still find the quotation and date in the note, because a similarity match is not a reading.

In [ ]:
r = run_task(request("P044", 3), ch04.planner_v2, context=ch04.context_v2)
print(r.hits, r.results["no_anticoag"])